# Ćwiczenie 4.1: ręczne Gini i pierwsze drzewo decyzyjne

To jest czysty notebook ćwiczeniowy dopasowany do wykładu `wyklad_4_17_05_2026.ipynb`.
Pracujemy na tym samym przykładzie `kup_lody`: najpierw widzimy drzewo jako punkt odniesienia, potem ręcznie liczymy, dlaczego właśnie takie splity są wybierane.

**Tryb pracy:** najpierw kartka / tablica / Markdown, potem kod.

Plik: **wersja dla studentów**.

Po notebooku student powinien umieć:

1. policzyć Gini w węźle,
2. policzyć ważone `Gini after` dla kilku splitów,
3. wybrać najlepszy split przez największy spadek nieczystości,
4. przełożyć drzewo na reguły `if/else`,
5. zakodować minimalne funkcje `gini` i `split_score`.


## 0. Dane z wykładu

Każdy wiersz to jedna sytuacja. Celem jest decyzja `kup_lody`.

| cecha | 1 oznacza | 0 oznacza |
|---|---|---|
| `cieplo` | jest ciepło | nie jest ciepło |
| `weekend` | jest weekend | dzień powszedni |
| `krotka_kolejka` | kolejka jest krótka | kolejka jest długa |

Uwaga: dane są wpisane bezpośrednio w notebook, żeby ćwiczenie działało bez dodatkowych plików CSV.


## Ważne: tutaj pytania są już binarne

W tym pierwszym notebooku wszystkie cechy są już zapisane jako `0/1`, czyli jako gotowe odpowiedzi **nie/tak**.

Przykład:

```text
cieplo = 1?          TAK / NIE
weekend = 1?         TAK / NIE
krotka_kolejka = 1?  TAK / NIE
```

Dlatego na tym etapie **nie szukamy progów**. Drzewo ma już gotowe kandydackie pytania i wybiera tylko to, które najbardziej zmniejsza nieczystość Gini.

Najważniejsze uproszczenie Notebooka 4.1:

```text
cecha binarna
      ↓
gotowe pytanie TAK/NIE
      ↓
Gini
      ↓
klasa: kup_lody = tak/nie
```

W kolejnym notebooku to uproszczenie zniknie: cechy będą liczbowe/punktowe, więc drzewo będzie musiało samo znaleźć progi typu `projekt_pkt <= 59.5`.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

from sklearn.tree import DecisionTreeClassifier, export_text, plot_tree
from sklearn.metrics import accuracy_score

pd.set_option("display.max_rows", 30)
pd.set_option("display.max_columns", 20)

lody = pd.DataFrame({
    "id": list(range(1, 15)),
    "cieplo": [0, 0, 0, 0, 1, 1, 1, 1, 1, 0, 1, 1, 1, 1],
    "weekend": [0, 0, 1, 1, 0, 0, 1, 1, 0, 1, 1, 0, 1, 0],
    "krotka_kolejka": [0, 1, 0, 1, 0, 1, 0, 1, 1, 1, 0, 0, 1, 1],
    "kup_lody": [
        "nie", "nie", "nie", "nie", "nie", "tak", "tak", "tak",
        "tak", "nie", "tak", "nie", "tak", "tak"
    ],
})

features = ["cieplo", "weekend", "krotka_kolejka"]
target = "kup_lody"

lody


In [ ]:
print("Liczba obserwacji:", len(lody))
print("Rozkład klas:")
display(lody[target].value_counts())


## 1. Drzewo jako punkt odniesienia

Najpierw generujemy drzewo tak jak na wykładzie. Potraktuj je jako rysunek, który trzeba teraz uzasadnić liczbami.

Najważniejsze pytania do ręcznego policzenia:

1. Dlaczego pierwszym pytaniem jest `cieplo <= 0.5`?
2. Dlaczego w ciepłej gałęzi pojawia się `krotka_kolejka <= 0.5`?
3. Skąd biorą się wartości `gini` i `samples` w węzłach?


In [ ]:
X = lody[features]
y = lody[target]

tree = DecisionTreeClassifier(criterion="gini", max_depth=3, random_state=42)
tree.fit(X, y)

print(export_text(tree, feature_names=features))

plt.figure(figsize=(10, 5))
plot_tree(
    tree,
    feature_names=features,
    class_names=list(tree.classes_),
    filled=True,
    rounded=True,
    impurity=True,
)
plt.title("Drzewo decyzyjne dla przykładu: kupić lody?")
plt.show()


### Doprecyzowanie: dlaczego `sklearn` pokazuje `<= 0.5` przy cechach binarnych?

W danych zapisaliśmy cechy jako `0/1`, czyli intuicyjnie myślimy o pytaniach:

```text
cieplo == 1?
weekend == 1?
krotka_kolejka == 1?
```

`DecisionTreeClassifier` traktuje jednak każdą cechę liczbowo. Dlatego dla cechy binarnej najwygodniejszy próg to `0.5`:

```text
cieplo <= 0.5      oznacza: cieplo = 0
cieplo > 0.5       oznacza: cieplo = 1
```

To jest dokładnie ten sam podział co pytanie `cieplo == 0/1`. Różnica jest tylko w zapisie.

**Wniosek dydaktyczny:** już w Notebooku 4.1 `sklearn` zapisuje podział jako próg, ale ponieważ cechy są binarne, ten próg jest banalny: `0.5`. Prawdziwe szukanie progów zaczyna się dopiero w Notebooku 4.2, gdy cecha może przyjmować wiele wartości liczbowych.

## 2. Wzory potrzebne do ręcznego liczenia

Dla klasyfikacji binarnej:

$$
Gini = 1 - p_{nie}^2 - p_{tak}^2
$$

Dla splitu liczymy ważoną nieczystość dzieci:

$$
Gini_{after} = \frac{N_L}{N}Gini_L + \frac{N_R}{N}Gini_R
$$

Zysk ze splitu:

$$
Gain = Gini_{before} - Gini_{after}
$$

W drzewie wybieramy split z najmniejszym `Gini_after`, czyli z największym `Gain`.


## Ćwiczenie A. Gini przed pierwszym podziałem

Uzupełnij ręcznie.

**Jak wypełnić krok po kroku:**

1. Policz, ile w całym zbiorze jest obserwacji z klasą `tak`.
2. Policz, ile w całym zbiorze jest obserwacji z klasą `nie`.
3. Policz proporcje klas:

   ```text
   p_tak = liczba_tak / liczba_wszystkich
   p_nie = liczba_nie / liczba_wszystkich
   ```

4. Wstaw proporcje do wzoru:

   ```text
   Gini = 1 - p_tak^2 - p_nie^2
   ```

W całym zbiorze mamy:

- `tak`: ...
- `nie`: ...
- razem: ...

$$
Gini_{before} = 1 - \left(\frac{...}{...}\right)^2 - \left(\frac{...}{...}\right)^2 = ...
$$


In [ ]:
# Krok pomocniczy: sprawdź liczności klas.
# To nie liczy jeszcze Gini — pokazuje tylko, ile jest przykładów klasy "tak" i "nie".
# Na podstawie tych liczności policz Gini ręcznie w następnej komórce.
lody[target].value_counts()

In [ ]:
# TODO A: po ręcznym liczeniu wpisz tutaj własny wynik Gini dla całego zbioru.
#
# Ta komórka ma zawierać wynik z kartki, a nie wynik skopiowany z funkcji Pythona.
#
# Pseudokod z kartki:
# 1. liczba_tak = liczba obserwacji z kup_lody == "tak"
# 2. liczba_nie = liczba obserwacji z kup_lody == "nie"
# 3. n = liczba_tak + liczba_nie
# 4. p_tak = liczba_tak / n
# 5. p_nie = liczba_nie / n
# 6. gini_before_recznie = 1 - p_tak**2 - p_nie**2
#
# Wpisz jedną liczbę, np. 0.1234, zamiast wielokropka.
gini_before_recznie = ...

gini_before_recznie

In [ ]:
# Komórka kontrolna — uruchom dopiero PO wpisaniu wyniku ręcznego powyżej.
#
# Ten kod liczy dokładnie to samo co wzór z kartki, ale automatycznie.
# Nie jest nowym zadaniem ani inną definicją Gini. To tylko sprawdzenie,
# czy wynik wpisany w gini_before_recznie zgadza się z obliczeniem kodowym.


def gini_z_licznosci_tak_nie(labels):
    # KROK 1. Policz liczności klas.
    counts = labels.value_counts()

    # KROK 2. Policz liczbę wszystkich obserwacji.
    n = counts.sum()

    # KROK 3. Policz proporcje klas tak/nie.
    p_tak = counts.get("tak", 0) / n
    p_nie = counts.get("nie", 0) / n

    # KROK 4. Zastosuj ten sam wzór co w obliczeniu ręcznym.
    return 1 - p_tak**2 - p_nie**2


gini_before_kodem = gini_z_licznosci_tak_nie(lody[target])

if gini_before_recznie is Ellipsis:
    print("Najpierw wpisz wynik ręczny w zmiennej gini_before_recznie.")
elif np.isclose(gini_before_recznie, gini_before_kodem):
    print("OK: wynik ręczny zgadza się z obliczeniem kodowym.")
else:
    print("Wynik ręczny nie zgadza się z obliczeniem kodowym.")
    print("Sprawdź: liczba_tak, liczba_nie, p_tak, p_nie oraz wzór 1 - p_tak**2 - p_nie**2.")

## Ćwiczenie B. Trzy kandydaty na pierwszy split

Porównujemy trzy pytania:

1. `cieplo == 0/1`,
2. `weekend == 0/1`,
3. `krotka_kolejka == 0/1`.

Dla każdej cechy policz:

- liczności klas w gałęzi `0`,
- liczności klas w gałęzi `1`,
- Gini każdej gałęzi,
- ważone `Gini_after`,
- `Gain`.

**Pseudokod dla każdego splitu:**

```text
DLA wybranej cechy, np. cieplo:
    1. podziel dane na dwie gałęzie: cecha = 0 oraz cecha = 1
    2. w każdej gałęzi policz klasy tak/nie
    3. w każdej gałęzi policz Gini
    4. policz wagę gałęzi = liczba_obserwacji_w_gałęzi / 14
    5. policz wkład gałęzi = waga_gałęzi * Gini_gałęzi
    6. Gini_after = suma wkładów obu gałęzi
    7. Gain = Gini_before - Gini_after
```

Najlepszy split to ten, który ma **najmniejsze `Gini_after`** albo równoważnie **największy `Gain`**.


In [ ]:
# Tabele liczności do ręcznej pracy.
# Wiersze: wartość cechy, kolumny: klasa docelowa.
for feature in features:
    print()
    print("CECHA:", feature)
    display(pd.crosstab(lody[feature], lody[target], margins=True))


### Split 1: `cieplo`

**Jak wypełniać:** najpierw korzystasz z tabeli liczności z komórki powyżej, potem osobno liczysz Gini dla gałęzi `0` i `1`.

Dla `cieplo = 0`:

- `nie`: ...
- `tak`: ...
- liczba obserwacji w gałęzi: ...
- $Gini_0 = 1 - (liczba\_nie/liczba\_gałęzi)^2 - (liczba\_tak/liczba\_gałęzi)^2 = ...$

Dla `cieplo = 1`:

- `nie`: ...
- `tak`: ...
- liczba obserwacji w gałęzi: ...
- $Gini_1 = 1 - (liczba\_nie/liczba\_gałęzi)^2 - (liczba\_tak/liczba\_gałęzi)^2 = ...$

Teraz ważymy oba wyniki licznością gałęzi:

$$
Gini_{after} = \frac{liczba\_gałęzi\_0}{14}\cdot Gini_0 + \frac{liczba\_gałęzi\_1}{14}\cdot Gini_1 = ...
$$

Na końcu liczymy zysk splitu:

$$
Gain = Gini_{before} - Gini_{after} = ...
$$


### Split 2: `weekend`

Wypełnij analogicznie jak dla `cieplo`.

**Pseudokod:**

```text
1. Weź tylko kolumny weekend oraz kup_lody.
2. Podziel obserwacje na weekend = 0 i weekend = 1.
3. W każdej gałęzi policz tak/nie.
4. Dla każdej gałęzi policz Gini.
5. Zważ Gini gałęzi przez liczebność gałęzi.
6. Zsumuj wkłady i policz Gain.
```

Dla `weekend = 0`:

- `nie`: ...
- `tak`: ...
- liczba obserwacji w gałęzi: ...
- $Gini_0 = ...$

Dla `weekend = 1`:

- `nie`: ...
- `tak`: ...
- liczba obserwacji w gałęzi: ...
- $Gini_1 = ...$

$$
Gini_{after} = \frac{...}{14}\cdot Gini_0 + \frac{...}{14}\cdot Gini_1 = ...
$$

$$
Gain = Gini_{before} - Gini_{after} = ...
$$


### Split 3: `krotka_kolejka`

Wypełnij analogicznie jak dla dwóch poprzednich cech.

**Pseudokod:**

```text
1. Podziel dane według krotka_kolejka = 0 oraz krotka_kolejka = 1.
2. W każdej gałęzi policz tak/nie.
3. Zamień liczności na proporcje.
4. Podstaw proporcje do wzoru Gini.
5. Policz ważone Gini_after.
6. Policz Gain.
```

Dla `krotka_kolejka = 0`:

- `nie`: ...
- `tak`: ...
- liczba obserwacji w gałęzi: ...
- $Gini_0 = ...$

Dla `krotka_kolejka = 1`:

- `nie`: ...
- `tak`: ...
- liczba obserwacji w gałęzi: ...
- $Gini_1 = ...$

$$
Gini_{after} = \frac{...}{14}\cdot Gini_0 + \frac{...}{14}\cdot Gini_1 = ...
$$

$$
Gain = Gini_{before} - Gini_{after} = ...
$$


### Wniosek po pierwszym poziomie

Uzupełnij tabelę, a potem uporządkuj cechy od najlepszego splitu do najsłabszego splitu.

**Jak podjąć decyzję:**

```text
1. Zbierz Gini_after dla wszystkich trzech cech.
2. Najlepsza cecha = najmniejsze Gini_after.
3. Dla kontroli policz Gain = Gini_before - Gini_after.
4. Najlepsza cecha powinna mieć największy Gain.
```

| cecha | Gini after | Gain | miejsce |
|---|---:|---:|---:|
| `cieplo` | ... | ... | ... |
| `weekend` | ... | ... | ... |
| `krotka_kolejka` | ... | ... | ... |

Najlepszy pierwszy split: `...`, ponieważ `...`.

**Wskazówka interpretacyjna:** nie patrz tylko na same wzory. Sprawdź też, czy split tworzy czystą gałąź oraz czy realnie rozdziela klasy `tak` i `nie`.



### Minićwiczenie kodowe po B. Tabela splitów z kodu

Po ręcznym policzeniu spróbuj napisać kod, który tworzy taką samą tabelę jak w Ćwiczeniu B.

Zadanie: dla jednej cechy zwróć tabelę z wierszami dla wartości `0` i `1`, a w kolumnach pokaż:

- liczność gałęzi,
- liczbę klas `tak` i `nie`,
- Gini gałęzi,
- wagę gałęzi,
- wkład gałęzi do `Gini_after`.

Na końcu zsumuj wkłady i porównaj wynik z ręcznymi obliczeniami.

**Pseudokod funkcji:**

```text
gini_from_counts(n_tak, n_nie):
    n = n_tak + n_nie
    p_tak = n_tak / n
    p_nie = n_nie / n
    return 1 - p_tak^2 - p_nie^2

split_summary_binary(data, feature):
    dla każdej wartości cechy, czyli 0 i 1:
        wybierz obserwacje z tej gałęzi
        policz n_tak, n_nie oraz n
        policz gini_value
        policz weight = n / n_total
        policz contribution = weight * gini_value
        dopisz wiersz do tabeli
    zwróć tabelę
```


In [ ]:

# Ćwiczenie B-kod: zautomatyzuj ręczne obliczenia z Ćwiczenia B.
#
# Cel: kod ma zrobić to samo, co wcześniej robiliśmy na kartce:
# policzyć Gini w obu gałęziach i ważone Gini_after dla jednej cechy.
#
# Uzupełniaj po jednej linijce. Najpierw uruchom dla jednej cechy, np. "cieplo",
# a dopiero potem dla wszystkich trzech cech.


def gini_from_counts(n_tak, n_nie):
    # n to liczba obserwacji w jednej gałęzi drzewa.
    n = n_tak + n_nie

    # Zabezpieczenie: jeśli gałąź byłaby pusta, przyjmujemy Gini = 0.
    # W tym ćwiczeniu gałęzie nie powinny być puste, ale to dobry nawyk.
    if n == 0:
        return 0

    # TODO 1: policz proporcję klasy "tak" w tej gałęzi.
    # Wzór z kartki:
    #     p_tak = liczba_tak / liczba_obserwacji_w_gałęzi
    p_tak = ...

    # TODO 2: policz proporcję klasy "nie" w tej gałęzi.
    # Wzór z kartki:
    #     p_nie = liczba_nie / liczba_obserwacji_w_gałęzi
    p_nie = ...

    # TODO 3: zwróć nieczystość Gini dla tej gałęzi.
    # Wzór:
    #     Gini = 1 - p_tak**2 - p_nie**2
    return ...


def split_summary_binary(data, feature, target="kup_lody"):
    rows = []
    n_total = len(data)

    # data.groupby(feature) tworzy osobne części danych dla feature=0 i feature=1.
    # value to wartość cechy, np. 0 albo 1.
    # part to podzbiór danych wpadający do tej gałęzi.
    for value, part in data.groupby(feature):
        counts = part[target].value_counts()
        n_tak = counts.get("tak", 0)
        n_nie = counts.get("nie", 0)
        n = len(part)

        # TODO 4: policz Gini tej konkretnej gałęzi.
        # Użyj funkcji napisanej wyżej:
        #     gini_from_counts(n_tak, n_nie)
        gini_value = ...

        # TODO 5: policz wagę gałęzi.
        # Waga mówi, jaka część całego zbioru trafiła do tej gałęzi:
        #     weight = liczba_obserwacji_w_gałęzi / liczba_obserwacji_w_całym_zbiorze
        weight = ...

        # TODO 6: policz wkład tej gałęzi do Gini_after.
        # Wzór:
        #     contribution = weight * gini_value
        # Potem suma contribution dla obu gałęzi daje Gini_after.
        contribution = ...

        rows.append({
            "cecha": feature,
            "wartosc": value,
            "n": n,
            "tak": n_tak,
            "nie": n_nie,
            "gini": gini_value,
            "waga": weight,
            "wklad_do_gini_after": contribution,
        })

    return pd.DataFrame(rows)


if gini_from_counts(1, 1) is Ellipsis:
    print("TODO: uzupełnij gini_from_counts i split_summary_binary.")
else:
    parent_counts = lody[target].value_counts()
    parent_gini_code = gini_from_counts(parent_counts.get("tak", 0), parent_counts.get("nie", 0))

    for feature in features:
        summary = split_summary_binary(lody, feature)
        display(summary.round(6))

        if summary["wklad_do_gini_after"].map(lambda x: x is Ellipsis).any():
            print(f"{feature}: TODO: uzupełnij obliczanie wkładu do Gini_after.")
        else:
            gini_after = summary["wklad_do_gini_after"].sum()
            print(f"{feature}: Gini_after={gini_after:.6f}, Gain={parent_gini_code - gini_after:.6f}")


In [ ]:
# Komórka pomocniczo-kontrolna po Ćwiczeniu B-kod.
#
# Funkcje poniżej liczą automatycznie to samo, co liczyliśmy ręcznie w Ćwiczeniu B.
# W notebooku studenckim traktuj je jako sprawdzenie i narzędzie do kolejnych sekcji,
# a nie jako zamiennik ręcznego rozwiązania.


def gini_reference(labels):
    # Wersja ogólna: działa dla dowolnych nazw klas, nie tylko "tak"/"nie".
    probs = labels.value_counts(normalize=True)
    return 1 - np.sum(probs ** 2)


def split_score_reference(data, feature, target=target):
    parent = gini_reference(data[target])
    rows = []
    weighted = 0.0

    for value, part in data.groupby(feature):
        child_gini = gini_reference(part[target])
        weight = len(part) / len(data)
        weighted += weight * child_gini
        rows.append({
            "cecha": feature,
            "wartosc": value,
            "n": len(part),
            "klasy": dict(part[target].value_counts()),
            "gini_dziecka": child_gini,
            "waga": weight,
        })

    return weighted, parent - weighted, pd.DataFrame(rows)


def b_kod_uzupelniony():
    """Sprawdza, czy minićwiczenie B-kod zostało uzupełnione na tyle, by pokazać kontrolę."""
    try:
        if gini_from_counts(1, 1) is Ellipsis:
            return False
        summary = split_summary_binary(lody, features[0])
        return not summary.map(lambda x: x is Ellipsis).any().any()
    except Exception:
        return False


if not b_kod_uzupelniony():
    print("Najpierw uzupełnij komórkę B-kod. Potem ta komórka pokaże tabelę kontrolną dla splitów.")
else:
    rows = []
    for feature in features:
        g_after, gain, _ = split_score_reference(lody, feature)
        rows.append({"cecha": feature, "gini_after": g_after, "gain": gain})

    display(pd.DataFrame(rows).sort_values("gini_after").round(6))

## Ćwiczenie C. Drugi poziom drzewa: tylko ciepłe dni

Po splicie `cieplo` gałąź `cieplo = 0` jest czysta: wszyscy mają `kup_lody = nie`.

Musimy więc dalej podzielić tylko gałąź:

```text
cieplo = 1
```

W tej gałęzi porównujemy już tylko:

- `weekend`,
- `krotka_kolejka`.


In [ ]:
warm = lody[lody["cieplo"] == 1].copy()

print("Liczba ciepłych dni:", len(warm))
display(warm)
print("Rozkład klas w ciepłej gałęzi:")
display(warm[target].value_counts())

for feature in ["weekend", "krotka_kolejka"]:
    print()
    print("CECHA W CIEPŁEJ GAŁĘZI:", feature)
    display(pd.crosstab(warm[feature], warm[target], margins=True))


### Gini dla ciepłej gałęzi

Teraz pracujemy tylko na obserwacjach, dla których `cieplo = 1`.

**Jak wypełnić:**

```text
1. Weź tylko ciepłe dni.
2. Policz w tej mniejszej grupie klasy tak/nie.
3. Policz proporcje klas.
4. Podstaw proporcje do wzoru Gini.
```

Uzupełnij:

- `tak`: ...
- `nie`: ...
- razem: ...

$$
Gini_{warm} = 1 - \left(\frac{...}{...}\right)^2 - \left(\frac{...}{...}\right)^2 = ...
$$


### Porównanie splitów w ciepłej gałęzi

Teraz oceniamy splity **tylko wewnątrz gałęzi `cieplo = 1`**. To ważne: nie liczymy już na całym zbiorze 14 obserwacji, tylko na podzbiorze ciepłych dni.

**Pseudokod:**

```text
DLA feature w [weekend, krotka_kolejka]:
    1. podziel warm na feature = 0 oraz feature = 1
    2. w każdej gałęzi policz tak/nie
    3. policz Gini każdej gałęzi
    4. policz ważone Gini_after, ale wagi licz względem liczby obserwacji w warm
    5. policz Gain = Gini_warm - Gini_after
```

Uzupełnij dla `weekend`:

- gałąź `0`: liczności `tak/nie` = ..., $Gini_0 = ...$
- gałąź `1`: liczności `tak/nie` = ..., $Gini_1 = ...$
- $Gini_{after} = ...$
- $Gain = ...$

Uzupełnij dla `krotka_kolejka`:

- gałąź `0`: liczności `tak/nie` = ..., $Gini_0 = ...$
- gałąź `1`: liczności `tak/nie` = ..., $Gini_1 = ...$
- $Gini_{after} = ...$
- $Gain = ...$

Lepszy drugi split: `...`, ponieważ `...`.


In [ ]:
rows = []
for feature in ["weekend", "krotka_kolejka"]:
    g_after, gain, details = split_score_reference(warm, feature)
    rows.append({"cecha": feature, "gini_after": g_after, "gain": gain})
    print()
    print(feature)
    display(details)

pd.DataFrame(rows).sort_values("gini_after").round(6)


## Ćwiczenie D. Trzeci poziom: domknięcie mieszanej gałęzi

Po splicie `cieplo = 1` oraz `krotka_kolejka = 0` zostaje mała mieszana gałąź.

Sprawdź, czy `weekend` domyka tę gałąź do czystych liści.


In [ ]:
mixed = lody[(lody["cieplo"] == 1) & (lody["krotka_kolejka"] == 0)].copy()

display(mixed)
print("Rozkład klas:")
display(mixed[target].value_counts())
print("Split po weekend:")
display(pd.crosstab(mixed["weekend"], mixed[target], margins=True))


### Trzeci poziom

Na tym etapie patrzymy tylko na małą mieszaną gałąź po wcześniejszych decyzjach.

**Pseudokod:**

```text
1. Weź obserwacje spełniające warunki poprzednich splitów.
2. Policz Gini przed kolejnym splitem.
3. Podziel tę małą gałąź według weekend = 0 oraz weekend = 1.
4. Policz Gini dzieci.
5. Sprawdź, czy oba dzieci są czyste, czyli czy Gini = 0.
```

Uzupełnij:

- Gini przed splitem w tej gałęzi: ...
- Gini dla `weekend = 0`: ...
- Gini dla `weekend = 1`: ...
- `Gini_after`: ...

Czy split po `weekend` domyka drzewo do czystych liści? ...


In [ ]:
g_after, gain, details = split_score_reference(mixed, "weekend")
print("Gini after:", round(g_after, 6))
print("Gain:", round(gain, 6))
details


## Ćwiczenie E. Zamiana drzewa na reguły `if/else`

Przepisz drzewo na reguły. Użyj kolejności z wygenerowanego drzewa.


Uzupełnij reguły:

```text
jeżeli cieplo == 0:
    kup_lody = ...
w przeciwnym razie:
    jeżeli krotka_kolejka == ...:
        kup_lody = ...
    w przeciwnym razie:
        jeżeli weekend == ...:
            kup_lody = ...
        w przeciwnym razie:
            kup_lody = ...
```

**Pseudokod decyzji:**

```text
1. Najpierw sprawdź pytanie z korzenia drzewa: cieplo.
2. Jeśli gałąź jest już czysta, od razu zwróć klasę.
3. Jeśli nie jest czysta, przejdź do następnego pytania: krotka_kolejka.
4. Jeśli nadal zostaje gałąź mieszana, użyj ostatniego pytania: weekend.
5. Każda ścieżka od korzenia do liścia powinna kończyć się etykietą "tak" albo "nie".
```


In [ ]:
def predict_lody_manual(row):
    # Funkcja dostaje jeden wiersz danych, czyli jedną sytuację.
    # Ma zwrócić dokładnie jedną etykietę: "tak" albo "nie".

    # KROK 1. Pierwsze pytanie drzewa: czy nie jest ciepło?
    # Jeśli row["cieplo"] == 0, idziemy do lewej gałęzi drzewa.
    if row["cieplo"] == 0:
        # TODO 1: wpisz klasę z liścia dla gałęzi cieplo == 0.
        # Zastąp "TODO" przez "tak" albo "nie".
        return "TODO"

    # KROK 2. Skoro jesteśmy tutaj, to znaczy, że cieplo == 1.
    # Teraz sprawdzamy kolejne pytanie drzewa: krotka_kolejka.
    if row["krotka_kolejka"] == 1:
        # TODO 2: wpisz klasę z liścia dla ciepłego dnia z krótką kolejką.
        return "TODO"

    # KROK 3. Skoro jesteśmy tutaj, to znaczy, że:
    # - cieplo == 1,
    # - krotka_kolejka == 0.
    # To jest ostatnia mieszana gałąź, więc sprawdzamy weekend.
    if row["weekend"] == 1:
        # TODO 3: wpisz klasę z liścia dla weekend == 1.
        return "TODO"

    # KROK 4. Ostatni przypadek: cieplo == 1, krotka_kolejka == 0, weekend == 0.
    # TODO 4: wpisz klasę z ostatniego liścia.
    return "TODO"


manual_pred = lody.apply(predict_lody_manual, axis=1)

if (manual_pred == "TODO").any():
    print("TODO: uzupełnij predict_lody_manual.")
else:
    sklearn_pred = tree.predict(X)
    check = lody[["id", "cieplo", "weekend", "krotka_kolejka", "kup_lody"]].copy()
    check["pred_reczna"] = manual_pred
    check["pred_sklearn"] = sklearn_pred
    check["zgodne"] = check["pred_reczna"] == check["pred_sklearn"]
    display(check)
    print("Accuracy reguł ręcznych:", accuracy_score(y, manual_pred))
    print("Zgodność ze sklearn:", np.mean(manual_pred == sklearn_pred))


## Ćwiczenie F. Zakoduj Gini i wybór splitu

Teraz zapisz mechanikę drzewa w kodzie. To nie jest jeszcze pełna implementacja drzewa, tylko najważniejszy krok: ocena jednego splitu.


In [ ]:
def gini_student(labels):
    # Funkcja dostaje same etykiety klas, np. kolumnę lody["kup_lody"].
    # Ma zwrócić jedną liczbę: Gini dla tego zbioru etykiet.

    # TODO 1: policz proporcje klas.
    # Podpowiedź:
    #     labels.value_counts(normalize=True)
    # zwróci np. proporcje klas "tak" i "nie".
    probs = ...

    # TODO 2: podnieś każdą proporcję do kwadratu.
    # Podpowiedź:
    #     probs ** 2
    squared_probs = ...

    # TODO 3: zsumuj kwadraty proporcji.
    # Podpowiedź:
    #     np.sum(squared_probs)
    sum_squared_probs = ...

    # TODO 4: zwróć Gini.
    # Wzór:
    #     Gini = 1 - suma_kwadratów_proporcji
    return ...


def split_score_student(data, feature, target="kup_lody"):
    # Funkcja ocenia jeden split, np. feature="cieplo".
    # Ma zwrócić ważone Gini_after po podziale na feature=0 i feature=1.

    # KROK 1. Przygotuj zmienną, do której będziemy dodawać wkłady gałęzi.
    weighted_gini = 0.0

    # KROK 2. Zapamiętaj liczbę obserwacji w aktualnym zbiorze.
    # Przyda się do liczenia wag gałęzi.
    n_total = len(data)

    # KROK 3. Przejdź po gałęziach splitu.
    # value to wartość cechy, np. 0 albo 1.
    # part to obserwacje, które trafiły do tej gałęzi.
    for value, part in data.groupby(feature):
        # TODO 5: policz Gini dziecka, czyli Gini etykiet w tej gałęzi.
        # Podpowiedź:
        #     gini_student(part[target])
        child_gini = ...

        # TODO 6: policz wagę dziecka.
        # Wzór:
        #     child_weight = liczba_obserwacji_w_gałęzi / liczba_obserwacji_w_rodzicu
        child_weight = ...

        # TODO 7: dodaj wkład tej gałęzi do sumy.
        # Wzór:
        #     weighted_gini = weighted_gini + child_weight * child_gini
        weighted_gini = ...

    # KROK 4. Po przejściu po obu gałęziach zwracamy Gini_after.
    return weighted_gini


student_gini = gini_student(lody[target])
if student_gini is Ellipsis:
    print("TODO: uzupełnij gini_student i split_score_student.")
else:
    print("gini_student(parent):", round(student_gini, 6))

    student_rows = []
    missing_split_score = False

    for feature in features:
        g_after = split_score_student(lody, feature)
        if g_after is Ellipsis:
            missing_split_score = True
            student_rows.append({
                "cecha": feature,
                "gini_after": "TODO",
                "gain": "TODO",
            })
        else:
            student_rows.append({
                "cecha": feature,
                "gini_after": g_after,
                "gain": student_gini - g_after,
            })

    if missing_split_score:
        print("TODO: uzupełnij split_score_student.")
        display(pd.DataFrame(student_rows))
    else:
        display(pd.DataFrame(student_rows).sort_values("gini_after").round(6))


## Mini-podsumowanie

Najważniejsza historia:

```text
1. Węzeł ma nieczystość Gini.
2. Split dzieli dane na dzieci.
3. Liczymy ważone Gini dzieci: Gini_after.
4. Wybieramy split, który najmocniej zmniejsza nieczystość.
5. Drzewo to seria takich lokalnych decyzji.
```

W tym notebooku pytania były proste, bo cechy były już binarne:

```text
cieplo = 1?  →  TAK / NIE
```

Naturalny następny krok to cechy liczbowe/punktowe. Wtedy drzewo nadal będzie robiło podział **TAK/NIE**, ale najpierw samo znajdzie próg:

```text
projekt_pkt <= 59.5?  →  TAK / NIE
```

Cała ścieżka materiału wygląda tak:

```text
Notebook 4.1
────────────
cechy binarne
      ↓
gotowe pytania TAK/NIE
      ↓
Gini
      ↓
klasa

Notebook 4.2A
─────────────
cechy liczbowe / punktowe
      ↓
progowanie: cecha <= próg?
      ↓
Gini
      ↓
klasa

Notebook 4.2B
─────────────
cechy liczbowe / punktowe
      ↓
progowanie: cecha <= próg?
      ↓
MSE / MAE
      ↓
liczba
```

Kluczowe zdanie do zapamiętania: **progowanie rozwiązuje problem cech liczbowych, ale samo w sobie nie oznacza jeszcze regresji**.
